# Notebook 3 — 3W Dataset Hodge Classification

This notebook compares four feature sets on the public
[3W dataset](https://github.com/petrobras/3W) for three-class flow-regime classification:

| Label | Regime | Count |
|---|---|---|
| 0 | Normal operation | 597 |
| 1 | Severe slugging | 106 |
| 2 | Flow instabilities | 344 |

## Feature sets compared

| Name | Features | Dimension |
|---|---|---|
| `baseline_scalar` | Persistence entropy, P∞, mean Betti (H₀ & H₁) | 6 |
| `hodge_only` | η_harm, η_grad, η_curl, harm/curl ratio, β₁, λ₁, spectral gap | 7 |
| `hodge_augmented` | baseline_scalar + hodge_only | 13 |

## Tools used

- **giotto-tda**: Takens embedding, VRP persistence, persistence entropy and Betti curves
- **Our Hodge module**: boundary matrices, decomposition, L₁ spectrum
- **scikit-learn**: RandomForest and LogisticRegression classifiers
- **GUDHI**: (optional) for cross-checking persistence diagrams via the simplex tree API

## Data setup

Download the 3W dataset and set `DATA_3W_ROOT` below:
```
git clone https://github.com/petrobras/3W   data/3w
```
The notebook expects directories like `data/3w/data/0/`, `data/3w/data/3/`, `data/3w/data/4/`.

## 1. Imports and paths

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path('..').resolve()))
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn import metrics

from gtda.time_series import TakensEmbedding
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceEntropy, BettiCurve, Amplitude

from hodge.boundary_matrices import get_rips_simplices, build_boundary_matrices, pressure_to_1cochain
from hodge.decomposition import hodge_decomposition
from hodge.spectrum import compute_l1_spectrum
from utils.epsilon_selection import epsilon_from_diagram
from classification.feature_sets import FEATURE_SETS, REGIME_LABELS

# ── Path configuration ──────────────────────────────────────────────────────
DATA_3W_ROOT  = pathlib.Path("../data/3w/data")
LABEL_DIRS    = {0: DATA_3W_ROOT / "0",
                 1: DATA_3W_ROOT / "3",
                 2: DATA_3W_ROOT / "4"}
SENSOR_COL    = "P-TPT"    # wellhead pressure
N_RESAMPLE    = 3000       # resample each time series to this length
PA_TO_BAR     = 1e-5

# Embedding parameters (from paper Section III)
EMBED_PARAMS  = {
    0: {"time_delay": 85,  "dimension": 9},   # normal
    1: {"time_delay": 125, "dimension": 7},   # slugging
    2: {"time_delay": 125, "dimension": 8},   # instabilities
}

## 2. Data loading and resampling

We load all CSV files for each class, interpolate missing values, and resample to a
uniform length of 3 000 points using periodic subsampling.  This makes all time series
comparable regardless of their original length.

In [ ]:
from gtda.time_series import Resampler

def load_class(label_dir: pathlib.Path, label: int, n: int = N_RESAMPLE):
    """Load all CSVs in a directory, resample to n points, return (signals, labels)."""
    signals = []
    for f in sorted(label_dir.glob("*.csv")):
        df = pd.read_csv(f).interpolate()
        if SENSOR_COL not in df.columns:
            continue
        period = max(1, int(len(df) / n))
        resampler = Resampler(period=period)
        _, sig = resampler.fit_transform_resample(df.index, df[SENSOR_COL])
        signals.append(sig * PA_TO_BAR)
    return signals, [label] * len(signals)

USE_SYNTHETIC = not DATA_3W_ROOT.exists()

if USE_SYNTHETIC:
    print("3W data not found — generating synthetic placeholder dataset")
    rng = np.random.default_rng(0)
    all_signals, all_labels = [], []
    for label, (n_samples, amp, freq) in [(0, 200, 0.05, 0), (1, 50, 3.0, 1/720), (2, 100, 0.5, 1/120)]:
        for _ in range(n_samples):
            t = np.arange(N_RESAMPLE)
            s = 164 + rng.normal(0, amp, N_RESAMPLE)
            if freq > 0:
                s += amp * np.sin(2 * np.pi * freq * t)
            all_signals.append(s)
            all_labels.append(label)
else:
    all_signals, all_labels = [], []
    for label, d in LABEL_DIRS.items():
        sigs, labs = load_class(d, label)
        all_signals.extend(sigs)
        all_labels.extend(labs)

all_signals = np.array(all_signals)
all_labels  = np.array(all_labels)

for lab, name in REGIME_LABELS.items():
    n = (all_labels == lab).sum()
    print(f"  {name:25s}: {n:4d} samples")
print(f"Total: {len(all_labels)} samples × {N_RESAMPLE} time steps each")

## 3. Feature extraction

We extract features in two stages:

### Stage A — Persistence features (giotto-tda)
For each time series: embed → compute VRP persistence diagram → extract scalars
(persistence entropy, max persistence / Amplitude, mean Betti curve).

### Stage B — Hodge features (our module)
For each time series: use the *same* embedding → compute ε from diagram → build
simplicial complex → Hodge decomposition + L₁ spectrum.

Both stages share the Takens embedding, so the Hodge analysis is directly informed
by the topological scale identified in Stage A.

In [ ]:
def extract_features_one(signal, embed_params):
    """Extract all features from one time series."""
    # ── Takens embedding ─────────────────────────────────────────────────
    te = TakensEmbedding(**embed_params)
    cloud = te.fit_transform(signal.reshape(1, -1))[0]

    # ── Persistence (giotto-tda) ─────────────────────────────────────────
    VRP  = VietorisRipsPersistence(homology_dimensions=[0, 1], infinity_values=1e10)
    diag = VRP.fit_transform(cloud[np.newaxis])[0]
    diag_3d = diag[np.newaxis]

    PE   = PersistenceEntropy(nan_fill_value=0.0)
    AMP  = Amplitude(metric="wasserstein", order=None)
    BC   = BettiCurve(n_bins=10)

    ent   = PE.fit_transform(diag_3d)[0]
    amp   = AMP.fit_transform(diag_3d)[0]
    betti = BC.fit_transform(diag_3d)[0]

    pers_feats = {
        "entropy_h0": float(ent[0]),  "entropy_h1": float(ent[1]),
        "p_inf_h0":   float(amp[0]),  "p_inf_h1":   float(amp[1]),
        "mean_betti0": float(np.mean(betti[0])),
        "mean_betti1": float(np.mean(betti[1])),
    }

    # ── Hodge features ───────────────────────────────────────────────────
    eps = epsilon_from_diagram(diag, strategy="most_persistent")
    zero_hodge = dict(eta_harm=0., eta_grad=0., eta_curl=0.,
                      harm_curl_ratio=0., beta1_hodge=0,
                      lambda1=0., lambda2=0., spectral_gap=0.)
    if eps is None:
        return {**pers_feats, **zero_hodge}

    sc = get_rips_simplices(cloud, eps, max_dim=2)
    if len(sc.edges) < 2 or len(sc.triangles) == 0:
        return {**pers_feats, **zero_hodge}

    B1, B2 = build_boundary_matrices(sc)
    n_nodes = len(sc.nodes)
    f1 = pressure_to_1cochain(signal[:n_nodes], sc)
    decomp = hodge_decomposition(f1, B1, B2)
    spec   = compute_l1_spectrum(B1, B2, n_eigs=min(15, len(sc.edges) - 2))

    hodge_feats = {
        "eta_harm": decomp.eta_harm, "eta_grad": decomp.eta_grad,
        "eta_curl": decomp.eta_curl, "harm_curl_ratio": decomp.harm_curl_ratio,
        "beta1_hodge": spec.beta1_hodge,
        "lambda1": spec.lambda1, "lambda2": spec.lambda2,
        "spectral_gap": spec.spectral_gap,
    }
    return {**pers_feats, **hodge_feats}


print("Extracting features (parallel)… this may take several minutes.")
results = Parallel(n_jobs=-1, verbose=5)(
    delayed(extract_features_one)(sig, EMBED_PARAMS[lbl])
    for sig, lbl in zip(all_signals, all_labels)
)
features_df = pd.DataFrame(results)
print(f"Feature matrix: {features_df.shape}")
features_df.head()

## 4. Classification comparison

We use **10-fold stratified cross-validation** with two classifiers:
- **LogisticRegression** (saga solver, L2 penalty, standardised features)
- **RandomForestClassifier** (100 trees, no scaling needed)

Each feature set is evaluated independently.

In [ ]:
X_all = features_df.values
y     = all_labels

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results_table = []
for fs_name, fs_cols in FEATURE_SETS.items():
    # Select columns that exist in the feature matrix
    available = [c for c in fs_cols if c in features_df.columns]
    X = features_df[available].values

    for clf_name, clf in [
        ("LogReg",        make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, solver='saga'))),
        ("RandomForest",  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ]:
        cv_res = cross_validate(clf, X, y, cv=cv,
                                scoring=["accuracy","f1_macro","recall_macro"],
                                return_train_score=False)
        results_table.append({
            "feature_set": fs_name,
            "classifier":  clf_name,
            "accuracy":    cv_res["test_accuracy"].mean(),
            "f1_macro":    cv_res["test_f1_macro"].mean(),
            "recall_macro":cv_res["test_recall_macro"].mean(),
        })

results_table = pd.DataFrame(results_table)
print(results_table.to_string(index=False, float_format="%.4f"))

## 5. Confusion matrix for best model

In [ ]:
from sklearn.model_selection import train_test_split

best_row = results_table.loc[results_table["f1_macro"].idxmax()]
print(f"Best: {best_row['classifier']} on '{best_row['feature_set']}' — F1={best_row['f1_macro']:.4f}")

best_cols = [c for c in FEATURE_SETS[best_row["feature_set"]] if c in features_df.columns]
X_best = features_df[best_cols].values

X_tr, X_te, y_tr, y_te = train_test_split(X_best, y, test_size=0.35, random_state=42, stratify=y)

if best_row["classifier"] == "RandomForest":
    best_clf = RandomForestClassifier(n_estimators=100, random_state=42)
else:
    best_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, solver='saga'))

best_clf.fit(X_tr, y_tr)
y_pred = best_clf.predict(X_te)

cm   = metrics.confusion_matrix(y_te, y_pred)
disp = metrics.ConfusionMatrixDisplay(cm, display_labels=[REGIME_LABELS[i] for i in sorted(REGIME_LABELS)])
disp.plot(cmap='Blues')
plt.title(f"Best model: {best_row['classifier']} / {best_row['feature_set']}")
plt.tight_layout()
plt.show()

print(metrics.classification_report(y_te, y_pred, target_names=[REGIME_LABELS[i] for i in sorted(REGIME_LABELS)]))

## 6. Feature importance (Random Forest on hodge_augmented)

The Random Forest's Gini impurity measure tells us which features contribute most to
the classification decision. We expect `η_harm` and `spectral_gap` to rank highly
alongside the persistence-based features.

In [ ]:
aug_cols = [c for c in FEATURE_SETS["hodge_augmented"] if c in features_df.columns]
X_aug = features_df[aug_cols].values

rf_aug = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_aug.fit(*train_test_split(X_aug, y, test_size=0.35, random_state=42, stratify=y)[:2])

imp = pd.Series(rf_aug.feature_importances_, index=aug_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
imp.plot.barh(ax=ax, color='steelblue')
ax.axvline(1 / len(aug_cols), ls='--', color='gray', lw=0.8, label='Uniform baseline')
ax.set_xlabel("Gini importance")
ax.set_title("Feature importances — hodge_augmented (Random Forest)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()